<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260316_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##< Node.js Express 라우팅 및 URL 변수 설정 >

In [1]:
import os
os.makedirs("/content/templates", exist_ok=True)

In [11]:
%%writefile /content/server1.js

const express = require('express');
const path = require('path');
const app = express();
const port = 3000;

// 템플릿 파일이 위치한 경로 정의
const templatePath = path.join(__dirname, 'templates');

// 메인 라우트: index.html 반환
app.get('/', (req, res) => {
    res.sendFile(path.join(templatePath, 'index.html'));
});

/**
 * 예제 1: 단일 Route Parameter
 */
app.get('/user/:userName', (req, res) => {
    res.send('성함: ' + req.params.userName);
});

/**
 * 예제 2: 다중 Route Parameters
 */
app.get('/delivery/:sender/:receiver', (req, res) => {
    const { sender, receiver } = req.params;
    res.send('발송인: ' + sender + ', 수취인: ' + receiver);
});

/**
 * 예제 3: Route Parameter 값 검증 (숫자 전용)
 * 최신 버전의 PathError를 피하기 위해 핸들러 내부에서 정규식으로 검증합니다.
 */
app.get('/id/:number', (req, res) => {
    const { number } = req.params;

    // 숫자가 아닌 값이 들어올 경우 처리
    if (!/^\d+$/.test(number)) {
        return res.status(400).send('에러: 숫자로 된 번호만 입력 가능합니다.');
    }

    res.send('승인 번호: ' + number);
});

/**
 * 예제 4: Routing 우선순위 (정적 경로를 변수 경로보다 먼저 정의)
 */
app.get('/member/regist', (req, res) => {
    res.send('회원 가입 화면입니다.');
});

app.get('/member/:name', (req, res) => {
    res.send('회원 성함: ' + req.params.name);
});

/**
 * 예제 5: Route Parameters와 Query Strings의 조합
 */
app.get('/search/:category', (req, res) => {
    const category = req.params.category;
    const keyword = req.query.keyword;
    res.send(category + ' 카테고리 내 ' + keyword + ' 검색 결과입니다.');
});

app.listen(port, () => {
    console.log('서버 실행 중: http://localhost:3000');
});

Writing /content/server1.js


In [5]:
%%writefile /content/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>Express 실습</title>
</head>
<body>
    <h1>Express Routing 실습 (Tree 구조)</h1>
    <hr>
    <ul>
        <li><a href="/user/김철수">1. 단일 변수 확인 (김철수)</a></li>
        <li><a href="/delivery/박영희/이민수">2. 다중 변수 확인 (박영희 -> 이민수)</a></li>
        <li><a href="/id/2026">3. 숫자 제약 확인 (2026)</a></li>
        <li><a href="/member/regist">4-1. 정적 경로 확인 (가입)</a></li>
        <li><a href="/member/최지우">4-2. 변수 경로 확인 (최지우)</a></li>
        <li><a href="/search/음식?keyword=비빔밥">5. 변수와 쿼리 조합 (음식/비빔밥)</a></li>
    </ul>
</body>
</html>

Overwriting /content/templates/index.html


In [6]:
%%bash
cd /content
npm init -y
npm install express
npm install -g cloudflared

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "server.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1",
    "start": "node server.js"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 65 packages, and audited 66 packages in 6s

22 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

added 1 package in 3s


In [7]:
%%bash
cd /content
nohup node server.js > /content/server.log 2>&1 &
sleep 2
cat /content/server.log

서버 실행 중: http://localhost:3000


In [8]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 5
cat /content/tunnel.log | grep trycloudflare.com

2026-03-16T06:12:25Z INF Requesting new quick Tunnel on trycloudflare.com...


In [9]:
%%bash
cat /content/tunnel.log | grep trycloudflare.com

2026-03-16T06:12:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-16T06:12:32Z INF |  https://comes-toolbox-sheer-equipped.trycloudflare.com                                    |




---



# Node.js Express Template Engine

## 1. Template Engine의 개념과 역할

Template Engine은 서버에서 보낸 데이터와 정적인 HTML 틀을 결합하여 최종적인 HTML 문서를 생성하는 도구입니다.

- **개념**: 정적인 HTML 코드 안에 서버의 데이터를 삽입할 수 있는 특수한 문법을 제공합니다.
- **역할**: 서버 측에서 렌더링을 수행하여 클라이언트에게 완성된 페이지를 전달합니다 (Server-Side Rendering).
- **중요성**: 게시판처럼 데이터가 수시로 변하는 서비스에서 하나의 HTML 틀로 수많은 콘텐츠를 동적으로 생성할 수 있게 합니다.

## 2. 주요 메커니즘: 데이터 보간 (Interpolation)

Template Engine은 서버의 데이터를 HTML에 박아 넣기 위해 고유한 문법을 사용합니다. 가장 대표적인 형식이 {{ }} 구문입니다.

- **표기법**: {{변수명}}
- **의미**: 해당 위치에 서버가 전달한 변수의 값을 치환하여 넣으라는 명령입니다.
- **작동 원리**: Express에서 res.render 함수를 호출할 때 전달된 객체의 키(key)와 템플릿의 변수명을 매칭합니다.

## 3. 게시판 구현을 위한 핵심 기능

### 3-1. 반복문 처리 (Iteration)

게시판 목록과 같이 배열 형태의 데이터를 출력할 때 사용합니다.

- **Handlebars 예시**: {{#each list}} ... {{/each}}
- **역할**: DB에서 가져온 게시글 목록 배열을 순회하며 반복적인 HTML 구조를 생성합니다.

### 3-2. 조건문 처리 (Conditionals)

데이터의 상태에 따라 화면 구성을 변경할 때 사용합니다.

- **Handlebars 예시**: {{#if isAdmin}} ... {{/if}}
- **역할**: 로그인 여부나 권한에 따라 글쓰기 버튼을 노출하거나 숨기는 등의 처리를 수행합니다.

## 4. Route Parameters와의 연동 Flow

게시판 상세 페이지를 예로 들어 Routing과 Template Engine이 결합하는 과정을 설명합니다.

1. **요청 수신**: 클라이언트가 /post/10 경로로 접속합니다.
2. **변수 캡처**: Express가 10을 Route Parameter인 :id 로 인식합니다.
3. **데이터 조회**: 서버가 id가 10인 게시글 데이터를 준비합니다.
4. **렌더링**: res.render('detail', { title: '공지', content: '내용' })를 통해 데이터를 템플릿에 전달합니다.
5. **응답**: Template Engine이 {{title}} 등을 실제 데이터로 치환한 최종 HTML을 브라우저에 전송합니다.



---



# 게시판을 이용한 Database Application 이해

## 1. 실습 목표

이번 실습의 목표는 사용자가 게시판 화면에서 글을 작성하고, 저장된 글 목록을 확인하며, 특정 글의 상세 내용을 조회하고, 기존 글을 수정하거나 삭제하는 전체 흐름을 이해하는 것이다.

즉, 이번 실습은 다음 흐름을 익히는 것이 핵심이다.

**게시글 작성 → DB 저장 → 목록 조회 → 상세 조회 → 수정 → 삭제**

---

## 2. 실습에서 학습할 핵심 개념

이번 실습에서 이해해야 할 핵심 개념은 다음과 같다.

- **HTML form** : 사용자가 입력한 값을 서버로 보내기 위한 구조
- **GET 방식** : 목록 조회, 상세 조회, 수정 화면 요청에 사용
- **POST 방식** : 글 저장, 글 수정, 글 삭제 처리에 사용
- **Express 라우팅** : URL에 따라 서버가 서로 다른 동작을 수행하는 구조
- **`express.urlencoded()`** : form 데이터 해석을 위한 설정
- **MariaDB INSERT / SELECT / UPDATE / DELETE** : 게시판 전기능 CRUD 처리
- **템플릿 파일 읽기** : HTML 파일을 읽고 필요한 내용만 치환하는 방식
- **화면과 처리 로직의 분리** : HTML은 templates, 처리는 server.js

## 3. 실습 시나리오

사용자가 게시판 목록 화면에 접속한다.

글쓰기 버튼을 눌러 제목, 작성자, 내용을 입력한다.

글쓰기 버튼을 누르면 `POST /board/write`로 데이터가 전달된다.

서버는 전달받은 값을 `board` 테이블에 저장한다.

목록에서 제목을 클릭하면 상세보기 화면으로 이동한다.

상세보기 화면에서는 수정 또는 삭제를 수행할 수 있다.

수정 버튼을 누르면 수정 화면으로 이동하고, 수정 완료 시 DB에 반영된다.

삭제 버튼을 누르면 해당 게시글이 삭제되고 목록으로 돌아간다.

In [ ]:
%%bash
pkill -f node; pkill -f cloudflared; echo "done"

In [1]:
%%bash
apt install -y mariadb-server mariadb-client -q
service mariadb start
mysql -u root -e "SELECT 1;"

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  galera-4 gawk libcgi-fast-perl libcgi-pm-perl libclone-perl
  libconfig-inifiles-perl libdbd-mysql-perl libdbi-perl libencode-locale-perl
  libfcgi-bin libfcgi-perl libfcgi0ldbl libhtml-parser-perl
  libhtml-tagset-perl libhtml-template-perl libhttp-date-perl
  libhttp-message-perl libio-html-perl liblwp-mediatypes-perl libmariadb3
  libterm-readkey-perl liburi-perl liburing2 mariadb-client-10.6
  mariadb-client-core-10.6 mariadb-common mariadb-server-10.6
  mariadb-server-core-10.6
Suggested packages:
  gawk-doc libmldbm-perl libnet-daemon-perl libsql-statement-perl
  libdata-dump-perl libipc-sharedcache-perl libbusiness-isbn-perl libwww-perl
  mailx mariadb-test netcat-openbsd
The following NEW packages will be installed:
  galera-4 gawk libcgi-fast-perl libcgi-pm-perl libclone-perl
  libconfig-inifiles-perl libdbd-mysql-perl libdbi-perl libencode-l

In [2]:
%%bash
mysql -u root -e "
CREATE DATABASE IF NOT EXISTS testdb;
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;"

mysql -u root testdb -e "
CREATE TABLE IF NOT EXISTS board (
    board_id INT PRIMARY KEY AUTO_INCREMENT,
    title VARCHAR(200) NOT NULL,
    writer VARCHAR(50) NOT NULL,
    content TEXT NOT NULL,
    reg_date DATETIME DEFAULT CURRENT_TIMESTAMP,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);"

In [3]:
import os
os.makedirs("/content/project/templates", exist_ok=True)

In [4]:
%%bash
cd /content/project
npm init -y
npm install express mysql2
npm install -g cloudflared

Wrote to /content/project/package.json:

{
  "name": "project",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 76 packages, and audited 77 packages in 7s

24 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

added 1 package in 2s


In [5]:
%%writefile /content/project/templates/board-write.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 작성</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: #f2f4f6;
            font-family: Arial, sans-serif;
            color: #191f28;
        }

        .page {
            max-width: 760px;
            margin: 0 auto;
            padding: 48px 20px 80px;
        }

        .card {
            background: #ffffff;
            border-radius: 24px;
            padding: 32px;
            box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);
        }

        h1 {
            margin: 0 0 10px;
            font-size: 32px;
            font-weight: 700;
        }

        .sub-text {
            margin: 0 0 32px;
            font-size: 15px;
            color: #8b95a1;
        }

        .form-group {
            margin-bottom: 20px;
        }

        label {
            display: block;
            margin-bottom: 10px;
            font-size: 14px;
            font-weight: bold;
            color: #4e5968;
        }

        input,
        textarea {
            width: 100%;
            border: 1px solid #e5e8eb;
            border-radius: 16px;
            padding: 16px 18px;
            font-size: 15px;
            outline: none;
        }

        input:focus,
        textarea:focus {
            border-color: #3182f6;
            box-shadow: 0 0 0 4px rgba(49, 130, 246, 0.12);
        }

        textarea {
            min-height: 240px;
            resize: vertical;
            line-height: 1.6;
        }

        .button-row {
            display: flex;
            gap: 12px;
            margin-top: 28px;
        }

        .btn,
        .btn-secondary {
            flex: 1;
            padding: 16px 20px;
            border-radius: 16px;
            text-align: center;
            text-decoration: none;
            font-size: 15px;
            font-weight: bold;
            border: none;
            cursor: pointer;
        }

        .btn {
            background: #3182f6;
            color: #ffffff;
        }

        .btn:hover {
            background: #1b64da;
        }

        .btn-secondary {
            background: #f2f4f6;
            color: #4e5968;
        }

        .btn-secondary:hover {
            background: #e5e8eb;
        }
    </style>
</head>
<body>
    <div class="page">
        <div class="card">
            <h1>게시글 작성</h1>
            <p class="sub-text">제목, 작성자, 내용을 입력한 뒤 저장할 수 있습니다.</p>

            <form action="/board/write" method="post">
                <div class="form-group">
                    <label for="title">제목</label>
                    <input type="text" id="title" name="title" required>
                </div>

                <div class="form-group">
                    <label for="writer">작성자</label>
                    <input type="text" id="writer" name="writer" required>
                </div>

                <div class="form-group">
                    <label for="content">내용</label>
                    <textarea id="content" name="content" required></textarea>
                </div>

                <div class="button-row">
                    <button class="btn" type="submit">저장하기</button>
                    <a class="btn-secondary" href="/board/list">목록 보기</a>
                </div>
            </form>
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/board-write.html


In [6]:
%%writefile /content/project/templates/board-list.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시판 목록</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: #f2f4f6;
            font-family: Arial, sans-serif;
            color: #191f28;
        }

        .page {
            max-width: 1100px;
            margin: 0 auto;
            padding: 48px 20px 80px;
        }

        .header {
            margin-bottom: 24px;
        }

        h1 {
            margin: 0 0 10px;
            font-size: 34px;
            font-weight: 700;
        }

        .sub-text {
            margin: 0;
            color: #8b95a1;
            font-size: 15px;
        }

        .card {
            background: #ffffff;
            border-radius: 24px;
            padding: 28px;
            box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);
        }

        .toolbar {
            display: flex;
            justify-content: flex-end;
            margin-bottom: 20px;
        }

        .write-btn {
            display: inline-block;
            padding: 14px 18px;
            border-radius: 14px;
            background: #3182f6;
            color: #ffffff;
            text-decoration: none;
            font-size: 14px;
            font-weight: bold;
        }

        .write-btn:hover {
            background: #1b64da;
        }

        table {
            width: 100%;
            border-collapse: collapse;
        }

        thead th {
            background: #f9fafb;
            color: #6b7684;
            padding: 16px 14px;
            border-bottom: 1px solid #f2f4f6;
            font-size: 14px;
        }

        tbody td {
            padding: 16px 14px;
            border-bottom: 1px solid #f2f4f6;
            text-align: center;
            font-size: 14px;
            color: #333d4b;
        }

        .title-cell {
            text-align: left;
        }

        .title-cell a {
            color: #191f28;
            text-decoration: none;
            font-weight: bold;
        }

        .title-cell a:hover {
            color: #3182f6;
        }

        .action-link {
            display: inline-block;
            padding: 8px 12px;
            border-radius: 10px;
            text-decoration: none;
            font-size: 13px;
            font-weight: bold;
        }

        .view-link {
            background: #eef6ff;
            color: #3182f6;
        }

        .edit-link {
            background: #f2f4f6;
            color: #4e5968;
        }

        .delete-btn {
            padding: 8px 12px;
            border: none;
            border-radius: 10px;
            background: #ffe2e5;
            color: #d92d20;
            font-size: 13px;
            font-weight: bold;
            cursor: pointer;
        }

        .delete-btn:hover {
            background: #ffd3d8;
        }

        .empty-box {
            padding: 48px 20px;
            text-align: center;
            color: #8b95a1;
            font-size: 15px;
        }

        .inline-form {
            margin: 0;
        }
    </style>
</head>
<body>
    <div class="page">
        <div class="header">
            <h1>게시판</h1>
            <p class="sub-text">등록된 게시글을 확인하고 새 글을 작성할 수 있습니다.</p>
        </div>

        <div class="card">
            <div class="toolbar">
                <a class="write-btn" href="/board/write">글쓰기</a>
            </div>

            {{table_section}}
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/board-list.html


In [7]:
%%writefile /content/project/templates/board-detail.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 상세보기</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: #f2f4f6;
            font-family: Arial, sans-serif;
            color: #191f28;
        }

        .page {
            max-width: 860px;
            margin: 0 auto;
            padding: 48px 20px 80px;
        }

        .card {
            background: #ffffff;
            border-radius: 24px;
            padding: 32px;
            box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);
        }

        h1 {
            margin: 0 0 24px;
            font-size: 30px;
            line-height: 1.4;
        }

        .meta-box {
            display: flex;
            gap: 16px;
            flex-wrap: wrap;
            margin-bottom: 28px;
            padding: 18px 20px;
            background: #f9fafb;
            border-radius: 16px;
        }

        .meta-item {
            font-size: 14px;
            color: #4e5968;
        }

        .meta-item strong {
            color: #191f28;
            margin-right: 8px;
        }

        .content-box {
            min-height: 240px;
            padding: 24px;
            border: 1px solid #f2f4f6;
            border-radius: 18px;
            font-size: 15px;
            line-height: 1.8;
            white-space: pre-wrap;
            color: #333d4b;
        }

        .button-row {
            display: flex;
            gap: 12px;
            margin-top: 28px;
        }

        .btn,
        .btn-secondary,
        .btn-danger {
            flex: 1;
            padding: 15px 18px;
            border-radius: 16px;
            text-align: center;
            text-decoration: none;
            font-size: 15px;
            font-weight: bold;
            border: none;
            cursor: pointer;
        }

        .btn {
            background: #3182f6;
            color: #ffffff;
        }

        .btn:hover {
            background: #1b64da;
        }

        .btn-secondary {
            background: #f2f4f6;
            color: #4e5968;
        }

        .btn-secondary:hover {
            background: #e5e8eb;
        }

        .btn-danger {
            background: #f04452;
            color: #ffffff;
        }

        .btn-danger:hover {
            background: #d92d20;
        }

        form {
            flex: 1;
            margin: 0;
        }
    </style>
</head>
<body>
    <div class="page">
        <div class="card">
            <h1>{{title}}</h1>

            <div class="meta-box">
                <div class="meta-item"><strong>번호</strong>{{board_id}}</div>
                <div class="meta-item"><strong>작성자</strong>{{writer}}</div>
                <div class="meta-item"><strong>작성일</strong>{{created_at}}</div>
            </div>

            <div class="content-box">{{content}}</div>

            <div class="button-row">
                <a class="btn-secondary" href="/board/list">목록</a>
                <a class="btn" href="/board/edit/{{board_id}}">수정</a>
                <form action="/board/delete/{{board_id}}" method="post" onsubmit="return confirm('정말 삭제하시겠습니까?');">
                    <button class="btn-danger" type="submit">삭제</button>
                </form>
            </div>
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/board-detail.html


In [8]:
%%writefile /content/project/templates/board-edit.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 수정</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: #f2f4f6;
            font-family: Arial, sans-serif;
            color: #191f28;
        }

        .page {
            max-width: 760px;
            margin: 0 auto;
            padding: 48px 20px 80px;
        }

        .card {
            background: #ffffff;
            border-radius: 24px;
            padding: 32px;
            box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);
        }

        h1 {
            margin: 0 0 10px;
            font-size: 32px;
            font-weight: 700;
        }

        .sub-text {
            margin: 0 0 32px;
            font-size: 15px;
            color: #8b95a1;
        }

        .form-group {
            margin-bottom: 20px;
        }

        label {
            display: block;
            margin-bottom: 10px;
            font-size: 14px;
            font-weight: bold;
            color: #4e5968;
        }

        input,
        textarea {
            width: 100%;
            border: 1px solid #e5e8eb;
            border-radius: 16px;
            padding: 16px 18px;
            font-size: 15px;
            outline: none;
        }

        input:focus,
        textarea:focus {
            border-color: #3182f6;
            box-shadow: 0 0 0 4px rgba(49, 130, 246, 0.12);
        }

        textarea {
            min-height: 240px;
            resize: vertical;
            line-height: 1.6;
        }

        .button-row {
            display: flex;
            gap: 12px;
            margin-top: 28px;
        }

        .btn,
        .btn-secondary {
            flex: 1;
            padding: 16px 20px;
            border-radius: 16px;
            text-align: center;
            text-decoration: none;
            font-size: 15px;
            font-weight: bold;
            border: none;
            cursor: pointer;
        }

        .btn {
            background: #3182f6;
            color: #ffffff;
        }

        .btn:hover {
            background: #1b64da;
        }

        .btn-secondary {
            background: #f2f4f6;
            color: #4e5968;
        }

        .btn-secondary:hover {
            background: #e5e8eb;
        }
    </style>
</head>
<body>
    <div class="page">
        <div class="card">
            <h1>게시글 수정</h1>
            <p class="sub-text">기존 게시글 내용을 수정할 수 있습니다.</p>

            <form action="/board/edit/{{board_id}}" method="post">
                <div class="form-group">
                    <label for="title">제목</label>
                    <input type="text" id="title" name="title" value="{{title}}" required>
                </div>

                <div class="form-group">
                    <label for="writer">작성자</label>
                    <input type="text" id="writer" name="writer" value="{{writer}}" required>
                </div>

                <div class="form-group">
                    <label for="content">내용</label>
                    <textarea id="content" name="content" required>{{content}}</textarea>
                </div>

                <div class="button-row">
                    <button class="btn" type="submit">수정 완료</button>
                    <a class="btn-secondary" href="/board/detail/{{board_id}}">취소</a>
                </div>
            </form>
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/board-edit.html


In [9]:
%%writefile /content/project/templates/board-message.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>처리 결과</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: #f2f4f6;
            font-family: Arial, sans-serif;
            color: #191f28;
        }

        .page {
            max-width: 520px;
            margin: 0 auto;
            padding: 100px 20px;
        }

        .card {
            background: #ffffff;
            border-radius: 24px;
            padding: 40px 32px;
            text-align: center;
            box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);
        }

        h1 {
            margin: 0 0 14px;
            font-size: 30px;
            font-weight: 700;
        }

        p {
            margin: 0 0 28px;
            font-size: 16px;
            color: #4e5968;
            line-height: 1.6;
        }

        .btn {
            display: inline-block;
            padding: 14px 22px;
            border-radius: 16px;
            background: #3182f6;
            color: #ffffff;
            text-decoration: none;
            font-weight: bold;
        }

        .btn:hover {
            background: #1b64da;
        }
    </style>
</head>
<body>
    <div class="page">
        <div class="card">
            <h1>{{title}}</h1>
            <p>{{message}}</p>
            <a class="btn" href="/board/list">목록으로 이동</a>
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/board-message.html


In [10]:
%%writefile /content/project/server.js

const express = require('express');
const path = require('path');
const fs = require('fs');
const mysql = require('mysql2');

const app = express();
const PORT = 3000;

app.use(express.urlencoded({ extended: true }));

const conn = mysql.createConnection({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb',
    dateStrings: true
});

conn.connect((err) => {
    if (err) {
        console.error('DB 연결 실패:', err);
        return;
    }
    console.log('DB 연결 성공');
});

function escapeHtml(value) {
    if (value === null || value === undefined) {
        return '';
    }

    return String(value)
        .replace(/&/g, '&amp;')
        .replace(/</g, '&lt;')
        .replace(/>/g, '&gt;')
        .replace(/"/g, '&quot;')
        .replace(/'/g, '&#39;');
}

function readTemplate(fileName) {
    const filePath = path.join(__dirname, 'templates', fileName);
    return fs.readFileSync(filePath, 'utf8');
}

function renderTemplate(template, data) {
    let html = template;

    for (const key in data) {
        const pattern = new RegExp(`{{${key}}}`, 'g');
        html = html.replace(pattern, data[key]);
    }

    return html;
}

function sendMessagePage(res, title, message) {
    const template = readTemplate('board-message.html');
    const html = renderTemplate(template, {
        title: escapeHtml(title),
        message: escapeHtml(message)
    });
    res.send(html);
}

app.get('/', (req, res) => {
    res.redirect('/board/list');
});

app.get('/board/list', (req, res) => {
    const sql = `
        SELECT
            board_id,
            title,
            writer,
            DATE_FORMAT(created_at, '%Y-%m-%d %H:%i:%s') AS created_at
        FROM board
        ORDER BY board_id DESC
    `;

    conn.query(sql, (err, rows) => {
        if (err) {
            console.error('목록 조회 실패:', err);
            return sendMessagePage(res, '조회 실패', '게시글 목록 조회 중 오류가 발생했습니다.');
        }

        let tableSection = '';

        if (rows.length === 0) {
            tableSection = `<div class="empty-box">등록된 게시글이 없습니다.</div>`;
        } else {
            let rowsHtml = '';

            for (let i = 0; i < rows.length; i++) {
                rowsHtml += `
                    <tr>
                        <td>${rows[i].board_id}</td>
                        <td class="title-cell">
                            <a href="/board/detail/${rows[i].board_id}">${escapeHtml(rows[i].title)}</a>
                        </td>
                        <td>${escapeHtml(rows[i].writer)}</td>
                        <td>${rows[i].created_at}</td>
                        <td>
                            <a class="action-link view-link" href="/board/detail/${rows[i].board_id}">조회</a>
                        </td>
                        <td>
                            <a class="action-link edit-link" href="/board/edit/${rows[i].board_id}">수정</a>
                        </td>
                        <td>
                            <form class="inline-form" action="/board/delete/${rows[i].board_id}" method="post" onsubmit="return confirm('정말 삭제하시겠습니까?');">
                                <button class="delete-btn" type="submit">삭제</button>
                            </form>
                        </td>
                    </tr>
                `;
            }

            tableSection = `
                <table>
                    <thead>
                        <tr>
                            <th style="width: 80px;">번호</th>
                            <th>제목</th>
                            <th style="width: 140px;">작성자</th>
                            <th style="width: 180px;">작성일</th>
                            <th style="width: 90px;">조회</th>
                            <th style="width: 90px;">수정</th>
                            <th style="width: 90px;">삭제</th>
                        </tr>
                    </thead>
                    <tbody>
                        ${rowsHtml}
                    </tbody>
                </table>
            `;
        }

        const template = readTemplate('board-list.html');
        const html = renderTemplate(template, {
            table_section: tableSection
        });

        res.send(html);
    });
});

app.get('/board/write', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'board-write.html'));
});

app.post('/board/write', (req, res) => {
    const { title, writer, content } = req.body;

    const sql = `
        INSERT INTO board (title, writer, content, created_at)
        VALUES (?, ?, ?, CONVERT_TZ(UTC_TIMESTAMP(), '+00:00', '+09:00'))
    `;

    conn.query(sql, [title, writer, content], (err) => {
        if (err) {
            console.error('게시글 저장 실패:', err);
            return sendMessagePage(res, '저장 실패', '게시글 저장 중 오류가 발생했습니다.');
        }

        res.redirect('/board/list');
    });
});

app.get('/board/detail/:id', (req, res) => {
    const boardId = req.params.id;

    const sql = `
        SELECT
            board_id,
            title,
            writer,
            content,
            DATE_FORMAT(created_at, '%Y-%m-%d %H:%i:%s') AS created_at
        FROM board
        WHERE board_id = ?
    `;

    conn.query(sql, [boardId], (err, rows) => {
        if (err) {
            console.error('상세 조회 실패:', err);
            return sendMessagePage(res, '조회 실패', '게시글 조회 중 오류가 발생했습니다.');
        }

        if (rows.length === 0) {
            return sendMessagePage(res, '조회 실패', '해당 게시글이 존재하지 않습니다.');
        }

        const row = rows[0];
        const template = readTemplate('board-detail.html');
        const html = renderTemplate(template, {
            board_id: String(row.board_id),
            title: escapeHtml(row.title),
            writer: escapeHtml(row.writer),
            content: escapeHtml(row.content),
            created_at: row.created_at
        });

        res.send(html);
    });
});

app.get('/board/edit/:id', (req, res) => {
    const boardId = req.params.id;

    const sql = `
        SELECT
            board_id,
            title,
            writer,
            content
        FROM board
        WHERE board_id = ?
    `;

    conn.query(sql, [boardId], (err, rows) => {
        if (err) {
            console.error('수정 화면 조회 실패:', err);
            return sendMessagePage(res, '조회 실패', '수정 화면 조회 중 오류가 발생했습니다.');
        }

        if (rows.length === 0) {
            return sendMessagePage(res, '조회 실패', '해당 게시글이 존재하지 않습니다.');
        }

        const row = rows[0];
        const template = readTemplate('board-edit.html');
        const html = renderTemplate(template, {
            board_id: String(row.board_id),
            title: escapeHtml(row.title),
            writer: escapeHtml(row.writer),
            content: escapeHtml(row.content)
        });

        res.send(html);
    });
});

app.post('/board/edit/:id', (req, res) => {
    const boardId = req.params.id;
    const { title, writer, content } = req.body;

    const sql = `
        UPDATE board
        SET title = ?, writer = ?, content = ?
        WHERE board_id = ?
    `;

    conn.query(sql, [title, writer, content, boardId], (err, result) => {
        if (err) {
            console.error('게시글 수정 실패:', err);
            return sendMessagePage(res, '수정 실패', '게시글 수정 중 오류가 발생했습니다.');
        }

        if (result.affectedRows === 0) {
            return sendMessagePage(res, '수정 실패', '해당 게시글이 존재하지 않습니다.');
        }

        res.redirect('/board/detail/' + boardId);
    });
});

app.post('/board/delete/:id', (req, res) => {
    const boardId = req.params.id;

    const sql = `
        DELETE FROM board
        WHERE board_id = ?
    `;

    conn.query(sql, [boardId], (err, result) => {
        if (err) {
            console.error('게시글 삭제 실패:', err);
            return sendMessagePage(res, '삭제 실패', '게시글 삭제 중 오류가 발생했습니다.');
        }

        if (result.affectedRows === 0) {
            return sendMessagePage(res, '삭제 실패', '해당 게시글이 존재하지 않습니다.');
        }

        res.redirect('/board/list');
    });
});

app.listen(PORT, () => {
    console.log('=========================================');
    console.log(` 게시판 서버가 포트 ${PORT}에서 작동 중입니다.`);
    console.log('=========================================');
});

Writing /content/project/server.js


In [11]:
%%bash
cd /content/project
nohup node server.js > /content/project/server.log 2>&1 &
sleep 2
cat /content/project/server.log

 게시판 서버가 포트 3000에서 작동 중입니다.
DB 연결 성공


In [12]:
%%bash
npm install -g cloudflared


changed 1 package in 2s


In [13]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 5
cat /content/tunnel.log | grep trycloudflare.com

2026-03-16T06:34:02Z INF Requesting new quick Tunnel on trycloudflare.com...
<title>Worker threw exception | api.trycloudflare.com | Cloudflare</title>
            <p>You've requested a page on a website (api.trycloudflare.com) that is on the <a href="https://www.cloudflare.com/5xx-error-landing/" target="_blank">Cloudflare</a> network. An unknown error occurred while rendering the page.</p>
            <p><strong>If you are the owner of this website:</strong><br />refer to <a href="https://developers.cloudflare.com/workers/observability/errors/" target="_blank">Workers - Errors and Exceptions</a> and check Workers Logs for api.trycloudflare.com.</p>


In [38]:
%%bash
cat /content/tunnel.log | grep trycloudflare.com

2026-03-16T06:36:16Z INF Requesting new quick Tunnel on trycloudflare.com...
<title>Worker threw exception | api.trycloudflare.com | Cloudflare</title>
            <p>You've requested a page on a website (api.trycloudflare.com) that is on the <a href="https://www.cloudflare.com/5xx-error-landing/" target="_blank">Cloudflare</a> network. An unknown error occurred while rendering the page.</p>
            <p><strong>If you are the owner of this website:</strong><br />refer to <a href="https://developers.cloudflare.com/workers/observability/errors/" target="_blank">Workers - Errors and Exceptions</a> and check Workers Logs for api.trycloudflare.com.</p>




---



In [39]:
%%bash
pkill -f cloudflared; echo "done"

done


In [40]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 10
cat /content/tunnel.log | grep trycloudflare.com

2026-03-16T06:39:11Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-16T06:39:19Z INF |  https://realtors-true-logo-substantially.trycloudflare.com                                |


In [46]:
%%bash
mysql -u root testdb -e "SELECT * FROM board;"

board_id	title	writer	content	reg_date	created_at
1	안녕하세요	소라	월요일 좋아	2026-03-16 06:41:13	2026-03-16 15:41:13
3	졸리다	123	ㅁㄴㅇㄻㄴㅇㄻㄴㅇㄹ	2026-03-16 06:44:39	2026-03-16 15:44:39
